# HAWI: Ataque al LWE mediante QAOA

**Alumno:** Carlos Yanes Pérez

Implementación del algoritmo HAWI propuesto en Zheng et al. (2025). *Quantum-classical hybrid algorithm for solving the learning-with-errors problem on NISQ devices*. Communications Physics. DOI: [10.1038/s42005-025-02126-w](https://doi.org/10.1038/s42005-025-02126-w)

El notebook implementa el experimento del paper usando los mismos parámetros ($n=2$, $m=6$, $q=17$) para la instancia LWE, resuelta con QAOA de una capa sobre 5 cúbits (Supplementary Figure 2).

> El código ha sido refactorizado usando Claude Sonnet 4.6.

# Instalación e importaciones

In [ ]:
%pip install qiskit scipy numpy sympy -q

import numpy as np
from math import gcd
from fractions import Fraction
from scipy.optimize import minimize
from qiskit.quantum_info import Statevector
from qiskit.circuit import QuantumCircuit, Parameter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 3.8 MB/s eta 0:00:00


# Instancia LWE

Instancia LWE$(m, n, q, B)$ con $n=2$, $m=6$, $q=17$, $\sigma=1$,
tal que $b = As + e \bmod q$.

In [ ]:
n, m, q = 2, 6, 17
sigma    = 1
np.random.seed(0)
s     = np.random.randint(0, q, n)
A     = np.random.randint(0, q, (m, n))
e     = np.round(np.random.normal(0, sigma, m)).astype(int) % q
b_vec = (A @ s + e) % q

print("Instancia LWE")
print(f"Parámetros : n={n}, m={m}, q={q}")
print(f"Secreto s  : {s}")
print(f"Matriz A   :\n{A}")
print(f"Error e    : {e}")
print(f"Vector b   : {b_vec}")
print(f"Verif.     : As+e mod {q} = {(A@s+e)%q}  (= b?)")

Instancia LWE
Parámetros : n=2, m=6, q=17
Secreto s  : [12 15]
Matriz A   :
[[ 0  3]
 [ 3  7]
 [ 9  4]
 [ 6 12]
 [ 1  6]
 [ 7 14]]
Error e    : [1 0 0 0 1 0]
Vector b   : [12  5 15 14  1  5]
Verif.     : As+e mod 17 = [12  5 15 14  1  5]  (= b?)


# Preprocesamiento clásico

## Eliminación gaussiana (mod $q$)
Base del núcleo de $A^\top$ en $\mathbb{Z}_q$: vectores $\{w_i\}$ con $A^\top w_i = 0 \bmod q$.

In [ ]:
def find_kernel_mod_q(A, q):
    """Base del núcleo de A (n x m) en Z_q mediante eliminación gaussiana."""
    n_rows, m_cols = A.shape
    M = A.copy() % q
    pivot_cols = []; pivot_row = 0
    for col in range(m_cols):
        pivot = next((r for r in range(pivot_row, n_rows)
                      if gcd(int(M[r,col])%q, q)==1), -1)
        if pivot == -1: continue
        M[[pivot_row,pivot]] = M[[pivot,pivot_row]]
        inv = pow(int(M[pivot_row,col])%q, -1, q)
        M[pivot_row] = (M[pivot_row]*inv) % q
        for row in range(n_rows):
            if row != pivot_row:
                M[row] = (M[row] - int(M[row,col])*M[pivot_row]) % q
        pivot_cols.append(col); pivot_row += 1
        if pivot_row == n_rows: break
    free_cols = [c for c in range(m_cols) if c not in pivot_cols]
    kernel = []
    for fc in free_cols:
        v = np.zeros(m_cols, dtype=int); v[fc] = 1
        for r, pc in enumerate(pivot_cols): v[pc] = int(-M[r,fc]) % q
        kernel.append(v % q)
    return np.array(kernel, dtype=int)

W = find_kernel_mod_q(A.T, q)
print(f"Vectores kernel: {W.shape[0]}")
for i, w in enumerate(W):
    print(f"  w_{i} = {w}   A^T w mod {q} = {(A.T@w)%q}")

Vectores kernel: 4
  w_0 = [ 0 14  1  0  0  0]   A^T w mod 17 = [0 0]
  w_1 = [12 15  0  1  0  0]   A^T w mod 17 = [0 0]
  w_2 = [12 11  0  0  1  0]   A^T w mod 17 = [0 0]
  w_3 = [14  9  0  0  0  1]   A^T w mod 17 = [0 0]


## Reducción LLL

La matriz extendida $W' = [W \mid qI_m]$ asegura que la red contiene
todos los vectores necesarios para la decisión LWE.

In [ ]:
def lll_exact(B, delta=Fraction(3,4)):
    """
    LLL con fracciones.
    Garantiza que no hay errores de redondeo en los vectores de la base.
    """
    B  = [[Fraction(x) for x in row] for row in B]
    n  = len(B)
    d  = len(B[0])

    def dot(u, v): return sum(u[i]*v[i] for i in range(d))

    def gram_schmidt(B):
        Bstar = []; mu = [[Fraction(0)]*n for _ in range(n)]
        for i in range(n):
            Bstar_i = B[i][:]
            for j in range(i):
                denom = dot(Bstar[j], Bstar[j])
                mu[i][j] = dot(B[i], Bstar[j])/denom if denom else Fraction(0)
                Bstar_i = [Bstar_i[k] - mu[i][j]*Bstar[j][k] for k in range(d)]
            Bstar.append(Bstar_i)
        return Bstar, mu

    k, iters = 1, 0
    while k < n and iters < 100000:
        iters += 1
        Bstar, mu = gram_schmidt(B)
        for j in range(k-1, -1, -1):
            m_kj = int(mu[k][j] + Fraction(1,2)) if mu[k][j] >= 0 else int(mu[k][j] - Fraction(1,2))
            if m_kj:
                B[k] = [B[k][i] - Fraction(m_kj)*B[j][i] for i in range(d)]
                Bstar, mu = gram_schmidt(B)
        d_k   = dot(Bstar[k],   Bstar[k])
        d_km1 = dot(Bstar[k-1], Bstar[k-1])
        if not d_km1 or d_k >= (delta - mu[k][k-1]**2)*d_km1:
            k += 1
        else:
            B[k], B[k-1] = B[k-1][:], B[k][:]
            k = max(k-1, 1)
    return [[int(x) for x in row] for row in B]


# Extender W con q*I_m y reducir los primeros m vectores
W_ext   = np.vstack([W, q*np.eye(m, dtype=int)]).astype(int)
B_raw   = lll_exact(W_ext[:m].tolist())

# Filtrar ceros, ordenar por norma y verificar pertenencia al núcleo
B_lll = [np.array(b) for b in B_raw if np.linalg.norm(b) > 0.5]
B_lll.sort(key=lambda b: np.linalg.norm(b))

print("Base LLL reducida")
for i, b in enumerate(B_lll):
    check = (A.T @ b) % q
    ok    = "(Ok)" if np.all(check==0) else "(Fallo)"
    print(f"  b_{i} = {b}   ||b||={np.linalg.norm(b):.3f}   A^Tb mod {q}={check} {ok}")

Base LLL reducida
  b_0 = [ 0 -1 -1 -1  1  0]   ||b||=2.000   A^Tb mod 17=[0 0] (Ok)
  b_1 = [ 1  0 -1  1  0 -2]   ||b||=2.646   A^Tb mod 17=[0 0] (Ok)
  b_2 = [-2 -1  1  0  1 -1]   ||b||=2.828   A^Tb mod 17=[0 0] (Ok)
  b_3 = [ 2 -2  0  0 -1  1]   ||b||=3.162   A^Tb mod 17=[0 0] (Ok)
  b_4 = [ 1  0 -1  2  0  2]   ||b||=3.162   A^Tb mod 17=[0 0] (Ok)
  b_5 = [2 0 1 1 2 0]   ||b||=3.162   A^Tb mod 17=[0 0] (Ok)


# Hamiltoniano de Ising

Hamiltoniano $H_r$ (Ec. 4 del paper) con 1 cúbit por vector base.
La codificación $\hat{x}_j = (1+\sigma^z_j)/2 \in \{0,1\}$ da $N_q = m-1 = 5$ cúbits.

In [ ]:
def build_ising_hamiltonian(B_lll, r=0):
    """
    H_r = ||b_r + sum_{i!=r} x_i b_i||^2  con x_i = (sigma_z_i + 1)/2
    Expande a forma Ising: H = sum_{i<j} J_ij sigma_z_i sigma_z_j
                               + sum_i h_i sigma_z_i + c
    """
    B   = np.array(B_lll, dtype=float)
    b_r = B[r]
    idx = [i for i in range(len(B)) if i != r]
    n_q = len(idx)

    M     = np.array([[float(np.dot(B[idx[a]], B[idx[b]]))
                       for b in range(n_q)] for a in range(n_q)])
    v     = np.array([float(np.dot(b_r, B[idx[i]])) for i in range(n_q)])
    c     = float(np.dot(b_r, b_r))
    h     = np.zeros(n_q)
    J     = np.zeros((n_q, n_q))
    const = c

    for i in range(n_q):
        const += M[i,i]/4 + v[i]/2
        h[i]  += M[i,i]/4 + v[i]/2
        for j in range(n_q):
            if j != i:
                h[i]  += M[i,j]/4
                const += M[i,j]/4
    for i in range(n_q):
        for j in range(i+1, n_q):
            J[i,j] += M[i,j]/4
    const += sum(v)/2
    return h, J, const, idx, n_q


# Seleccionar r cuyo estado fundamental tenga menor energía positiva
# El estado fundamental de H_r es el vector más corto no nulo de la red
B_arr = np.array(B_lll, dtype=float)
best_r, best_E = 0, np.inf
for r_try in range(len(B_lll)):
    b_r_t = B_arr[r_try]
    other = [i for i in range(len(B_lll)) if i != r_try]
    # El estado x=[0,...,0] da v = b_r, que es no nulo si ||b_r|| > 0
    # Incluirlo en la búsqueda del mínimo
    for state in range(2**len(other)):
        x   = [(state>>i)&1 for i in range(len(other))]
        v_v = b_r_t + sum(x[i]*B_arr[other[i]] for i in range(len(other)))
        E   = float(np.dot(v_v, v_v))
        if 0 < E < best_E: best_E = E; best_r = r_try

r = best_r
h, J, c_const, idx, n_qubits = build_ising_hamiltonian(B_lll, r=r)
print(f"=== Hamiltoniano (r={r}, {n_qubits} cúbits) ===")
print(f"Constante c : {c_const:.3f}")
print(f"h (lineal)  : {h}")
print(f"J (cuad.)   :\n{np.round(J,3)}")

b_r    = B_arr[r]
others = [i for i in range(len(B_lll)) if i != r]

# Calcular energías para todos los estados
energies = []
for state in range(2**n_qubits):
    x   = [(state>>i)&1 for i in range(n_qubits)]
    v_v = b_r + sum(x[i]*B_arr[others[i]] for i in range(n_qubits))
    E   = float(np.dot(v_v, v_v))
    energies.append((E, state, np.round(v_v).astype(int)))
energies.sort()

# El estado fundamental es el de menor E > 0
E_fundamental = next(E for E, _, _ in energies if E > 0)

print(f"\nEstados de menor energía:")
for E, state, v_v in energies[:6]:
    x  = [(state>>i)&1 for i in range(n_qubits)]
    ok = "<- fundamental" if E == E_fundamental else ""
    print(f"  x={x}  E={E:.1f}  v={v_v}  {ok}")

=== Hamiltoniano (r=0, 5 cúbits) ===
Constante c : 16.250
h (lineal)  : [ 2.   -0.25  3.5   2.5   4.  ]
J (cuad.)   :
[[ 0.   -0.25  0.    0.    0.5 ]
 [ 0.    0.   -1.   -1.25 -0.25]
 [ 0.    0.    0.    1.    0.5 ]
 [ 0.    0.    0.    0.    0.75]
 [ 0.    0.    0.    0.    0.  ]]

Estados de menor energía:
  x=[0, 0, 0, 0, 0]  E=4.0  v=[ 0 -1 -1 -1  1  0]  <- fundamental
  x=[1, 0, 0, 0, 0]  E=11.0  v=[ 1 -1 -2  0  1 -2]  
  x=[0, 0, 0, 1, 0]  E=12.0  v=[ 1 -1 -2  1  1  2]  
  x=[0, 1, 0, 1, 0]  E=12.0  v=[-1 -2 -1  1  2  1]  
  x=[0, 1, 0, 0, 0]  E=14.0  v=[-2 -2  0 -1  2 -1]  
  x=[0, 0, 0, 0, 1]  E=14.0  v=[ 2 -1  0  0  3  0]  


# Circuito QAOA ($p=1$)

Circuito de una capa tal como aparece en Supplementary Figure 2 del paper:
$$U(\beta,\gamma) = e^{-i\beta\sum_j\sigma^x_j} \cdot e^{-i\gamma H}$$
con estado inicial $|\phi_0\rangle = |+\rangle^{\otimes n}$.

In [ ]:
def build_qaoa_p1(h, J, n_q):
    """Circuito QAOA p=1"""
    gamma = Parameter('γ')
    beta  = Parameter('β')
    qc    = QuantumCircuit(n_q)

    # Estado inicial: superposición uniforme
    for q_idx in range(n_q): qc.h(q_idx)

    # ── Capa de problema e^{-i*gamma*H} ──────────────────────────────────────
    # Terminos lineales RZ(2*gamma*h_i)
    for i in range(n_q):
        if abs(h[i]) > 1e-10:
            qc.rz(2*gamma*float(h[i]), i)
    # Terminos cuadraticos RZZ(2*gamma*J_ij)
    for i in range(n_q):
        for j in range(i+1, n_q):
            if abs(J[i,j]) > 1e-10:
                qc.cx(i, j)
                qc.rz(2*gamma*float(J[i,j]), j)
                qc.cx(i, j)

    # ── Capa de mezcla e^{-i*beta*sum X_j} ───────────────────────────────────
    for q_idx in range(n_q): qc.rx(2*beta, q_idx)

    return qc, gamma, beta

qc, gamma, beta = build_qaoa_p1(h, J, n_qubits)
print(f"Circuito QAOA p=1:")
print(f"  Qúbits     : {n_qubits}  (= m-1 = {m}-1)")
print(f"  Parámetros : 2 (γ, β)")
print(f"  Profundidad: {qc.decompose().depth()}")
print(f"  Puertas    : {qc.decompose().size()}")

Circuito QAOA p=1:
  Qúbits     : 5  (= m-1 = 6-1)
  Parámetros : 2 (γ, β)
  Profundidad: 21
  Puertas    : 39


# Optimización

Inicialización heurística (Ec. 11 del paper).

In [ ]:
def evaluate_energy(params_vals, qc, gamma, beta, h, J, n_q, c_const, sa=1.0):
    """
    E(beta, gamma) = <phi_0|U^dag H U|phi_0>
    sa: factor de escala para gamma (Supplementary Note 6).
    """
    g_val, b_val = params_vals[0]/sa, params_vals[1]
    bound = qc.assign_parameters({gamma: g_val, beta: b_val})
    sv    = np.abs(Statevector(bound).data)**2

    E = 0.0
    for state in range(2**n_q):
        if sv[state] < 1e-12: continue
        z     = np.array([1 if (state>>i)&1 else -1 for i in range(n_q)], dtype=float)
        H_val = c_const + float(z@h) + sum(J[i,j]*z[i]*z[j]
                                           for i in range(n_q)
                                           for j in range(i+1, n_q))
        E += sv[state] * H_val
    return float(E)


# Factor de escala (Supplementary Note 6)
sa = max(abs(J.max()), abs(h.max()), 1.0) * 10

# Inicialización heurística p=1: gamma_1 = -2pi/c * 1/2, beta_1 = pi/4
theta0_h = np.array([-np.pi * sa / c_const, np.pi/4])  # escalado
E0_h     = evaluate_energy(theta0_h, qc, gamma, beta, h, J, n_qubits, c_const, sa)
print(f"Init. heurística: γ={theta0_h[0]/sa:.4f}, β={theta0_h[1]:.4f}")
print(f"Energía inicial : {E0_h:.4f}")

# Optimización con gradient-free (COBYLA, como en el paper)
result = minimize(
    evaluate_energy, theta0_h,
    args=(qc, gamma, beta, h, J, n_qubits, c_const, sa),
    method='COBYLA',
    options={'maxiter': 1000, 'rhobeg': 0.3}
)
g_opt = result.x[0]/sa
b_opt = result.x[1]
print(f"\nOptimización completada:")
print(f"  γ* = {g_opt:.4f},  β* = {b_opt:.4f}")
print(f"  Energía final : {result.fun:.4f}")
print(f"  Evaluaciones  : {result.nfev}")

Init. heurística: γ=-0.1933, β=0.7854
Energía inicial : 28.0064

Optimización completada:
  γ* = -0.1929,  β* = 2.2160
  Energía final : 8.6469
  Evaluaciones  : 316


# Extracción y decisión LWE

Se extrae el vector corto $\mathbf{v}^*$ del estado de mayor probabilidad
y se calcula el producto interior $I_p = \langle\mathbf{v}^*, b\rangle_q$
para decidir si $b$ es una muestra LWE o aleatoria.

In [ ]:
# ── Distribución de probabilidades ───────────────────────────────────────────
bound_opt = qc.assign_parameters({gamma: g_opt, beta: b_opt})
probs_opt  = np.abs(Statevector(bound_opt).data)**2

B_arr  = np.array(B_lll, dtype=float)
b_r    = B_arr[r]
others = [i for i in range(len(B_lll)) if i != r]

print("=== Distribución de probabilidades (top 8 estados) ===")
print(f"  {'Estado':>8}  {'Prob':>8}  {'||v||':>6}  {'v'}  {'A^Tv mod q'}")
print("  " + "-"*70)

top_states  = np.argsort(probs_opt)[::-1][:8]
best_v      = None
best_norm   = np.inf

for state in top_states:
    x     = [(state>>i)&1 for i in range(n_qubits)]
    v_vec = b_r + sum(x[i]*B_arr[others[i]] for i in range(n_qubits))
    norm  = np.linalg.norm(v_vec)
    v_int = np.round(v_vec).astype(int)
    check = (A.T @ v_int) % q
    ok    = "(Ok)" if np.all(check==0) else "(Fallo)"
    if norm > 0.1 and norm < best_norm and np.all(check==0):
        best_norm = norm; best_v = v_int.copy()
    print(f"  {format(state,f'0{n_qubits}b'):>8}  {probs_opt[state]:>8.4f}  "
          f"{norm:>6.3f}  {v_int}  {check} {ok}")

# ── Decisión ─────────────────────────────────────────────────────────────────
print(f"\nDecisión LWE")
if best_v is not None:
    Ip    = int(np.dot(best_v, b_vec) % q)
    cota  = (q/(sigma*np.pi)) * np.sqrt(0.5*np.log(10))
    print(f"Vector corto v* = {best_v},  ||v*|| = {best_norm:.3f}")
    print(f"Cota requerida  : ||v*|| <= {cota:.3f}  ({'Ok' if best_norm<=cota else 'Fallo'})")
    print(f"A^T v* mod {q}    = {(A.T@best_v)%q}  (debe ser 0)")
    print(f"Ip = <v*,b> mod {q} = {Ip}")
    print(f"<v*,e> mod {q}    = {int(np.dot(best_v,e))%q}  (contribución del error)")
    Ib = q//4
    if Ip <= Ib or Ip >= q-Ib:
        print(f"\n-> Ip={Ip} ∈ [0,{Ib}]∪[{q-Ib},{q}]: distribución GAUSSIANA -> instancia LWE")
    else:
        print(f"\n-> Ip={Ip} ∈ ({Ib},{q-Ib}): distribución UNIFORME -> muestra aleatoria")
else:
    print("No se encontró vector corto válido en el núcleo de A^T.")

=== Distribución de probabilidades (top 8 estados) ===
    Estado      Prob   ||v||  v  A^Tv mod q
  ----------------------------------------------------------------------
     00000    0.3167   2.000  [ 0 -1 -1 -1  1  0]  [0 0] (Ok)
     00010    0.2664   3.742  [-2 -2  0 -1  2 -1]  [0 0] (Ok)
     01010    0.0886   3.464  [-1 -2 -1  1  2  1]  [0 0] (Ok)
     00011    0.0596   4.359  [-1 -2 -1  0  2 -3]  [0 0] (Ok)
     00001    0.0556   3.317  [ 1 -1 -2  0  1 -2]  [0 0] (Ok)
     00110    0.0508   4.243  [ 0 -4  0 -1  1  0]  [0 0] (Ok)
     01000    0.0279   3.464  [ 1 -1 -2  1  1  2]  [0 0] (Ok)
     01011    0.0238   4.123  [ 0 -2 -2  2  2 -1]  [0 0] (Ok)

Decisión LWE
Vector corto v* = [ 0 -1 -1 -1  1  0],  ||v*|| = 2.000
Cota requerida  : ||v*|| <= 5.806  (Ok)
A^T v* mod 17    = [0 0]  (debe ser 0)
Ip = <v*,b> mod 17 = 1
<v*,e> mod 17    = 1  (contribución del error)

-> Ip=1 ∈ [0,4]∪[13,17]: distribución GAUSSIANA -> instancia LWE
